# 🎥 Demo — Detección de residuos sobre imágenes

Usa el modelo entrenado (`best.pt`) para detectar residuos en imágenes del
**conjunto de test** (que el modelo nunca vio) y descargar los resultados.

> Ejecuta esto **después** del notebook de entrenamiento (o con esa sesión aún
> viva). Si la sesión se reinició, vuelve a correr la celda de descompresión del
> dataset del notebook `02_entrenamiento_colab.ipynb`.

## 1. Cargar el modelo entrenado

In [ ]:
# Si tu sesión de Colab sigue viva tras entrenar, esto es casi instantáneo.
!pip install ultralytics -q

import os
from ultralytics import YOLO

# Localizar best.pt (en la sesión actual o en tu Google Drive)
rutas = [
    "/content/runs/detect/residuos_yolo11n/weights/best.pt",
    "/content/drive/MyDrive/residuos_best.pt",
]
MODELO = next((p for p in rutas if os.path.exists(p)), None)
if MODELO is None:
    from google.colab import drive
    drive.mount("/content/drive")
    MODELO = next((p for p in rutas if os.path.exists(p)), None)

assert MODELO, "No encuentro best.pt. Re-entrena o sube residuos_best.pt a tu Drive."
print("✅ Modelo cargado desde:", MODELO)
model = YOLO(MODELO)

## 2. Detectar residuos en imágenes de test y mostrar resultados

In [ ]:
import glob, random
import matplotlib.pyplot as plt
from PIL import Image

TEST_DIR = "/content/dataset/test/images"
assert os.path.isdir(TEST_DIR), "No encuentro el test set. Re-ejecuta la celda de descompresión del notebook de entrenamiento."

SALIDA = "/content/predicciones"
os.makedirs(SALIDA, exist_ok=True)

imagenes = sorted(glob.glob(TEST_DIR + "/*.jpg"))
random.seed(7)
muestra = random.sample(imagenes, 9)

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for ax, img_path in zip(axes.flat, muestra):
    r = model.predict(img_path, conf=0.25, verbose=False)[0]
    anotada = r.plot()[..., ::-1]                 # BGR -> RGB
    Image.fromarray(anotada).save(os.path.join(SALIDA, "pred_" + os.path.basename(img_path)))
    clases = [model.names[int(c)] for c in r.boxes.cls.tolist()]
    ax.imshow(anotada)
    ax.set_title(f"{len(r.boxes)} detec: {', '.join(clases) or 'ninguna'}", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.savefig("/content/mosaico_predicciones.png", dpi=110)
plt.show()
print("Imágenes anotadas guardadas en:", SALIDA)

## 3. Descargar los resultados a tu PC

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/predicciones", "zip", "/content/predicciones")
files.download("/content/predicciones.zip")          # todas las imágenes anotadas
files.download("/content/mosaico_predicciones.png")  # el mosaico para el informe